In [1]:
import json

with open("chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 4 chunks


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Extract just the text from each chunk
texts = [chunk['text'] for chunk in chunks]

# TF-IDF: converts text into numbers based on word importance
# (words that are common across all documents get less weight;
# words that are distinctive to a specific document get more weight)
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(texts)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
# shape = (number of chunks, number of unique words across all documents)

TF-IDF matrix shape: (4, 276)


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_top_k(question, vectorizer, tfidf_matrix, chunks, k=3):
    """
    Given a question, find the top-k most similar chunks using TF-IDF + cosine similarity.
    """
    # Convert the question into the same TF-IDF space as the documents
    question_vector = vectorizer.transform([question])
    
    # Compare the question vector against every chunk vector
    similarities = cosine_similarity(question_vector, tfidf_matrix)[0]
    
    # Get indices of the top-k highest similarity scores
    top_indices = similarities.argsort()[::-1][:k]
    
    results = []
    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "similarity_score": round(float(similarities[idx]), 4)
        })
    
    return results

In [4]:
results = retrieve_top_k("What happens if I pay late?", vectorizer, tfidf_matrix, chunks, k=3)

for r in results:
    print(f"\nScore: {r['similarity_score']} | Source: {r['chunk']['source_document']}")
    print(r['chunk']['text'][:150] + "...")


Score: 0.1597 | Source: faq_orders_payment.txt
FREQUENTLY ASKED QUESTIONS: Order & Payment Process Q: How long does it take to process a new order? A: Standard orders are processed within 2-3 busin...

Score: 0.03 | Source: invoice_nov2024.txt
INVOICE #INV-2024-1187 Superstore Retail Group Billing Department Bill To: Regional Office - West Invoice Date: November 15, 2024 Due Date: December 1...

Score: 0.0 | Source: policy_discount_approval.txt
POLICY DOCUMENT: Discount Approval Policy Document ID: POL-SALES-004 Effective Date: January 1, 2024 Department: Sales Operations Purpose: This policy...


In [5]:
test_questions = [
    "What is the discount policy for furniture?",
    "How much was the total invoice amount?",
    "Which region performed best in October?"
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Question: {q}")
    results = retrieve_top_k(q, vectorizer, tfidf_matrix, chunks, k=1)
    top = results[0]
    print(f"Top match: {top['chunk']['source_document']} (score: {top['similarity_score']})")


Question: What is the discount policy for furniture?
Top match: policy_discount_approval.txt (score: 0.3314)

Question: How much was the total invoice amount?
Top match: invoice_nov2024.txt (score: 0.226)

Question: Which region performed best in October?
Top match: report_october2024.txt (score: 0.2553)


In [6]:
def ask_rag_extractive(question, vectorizer, tfidf_matrix, chunks, k=1):
    """
    A fully free RAG version: no LLM call, just return the most relevant
    chunk(s) as the 'answer', with the source clearly cited.
    """
    retrieved = retrieve_top_k(question, vectorizer, tfidf_matrix, chunks, k=k)
    
    if retrieved[0]['similarity_score'] < 0.05:
        return {
            "question": question,
            "answer": "I don't have enough information to answer that.",
            "sources": []
        }
    
    best = retrieved[0]
    return {
        "question": question,
        "answer": best['chunk']['text'],
        "sources": [best['chunk']['source_document']],
        "confidence_score": best['similarity_score']
    }

In [7]:
result = ask_rag_extractive("What is the discount policy for furniture?", vectorizer, tfidf_matrix, chunks)

print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")
print(f"Source: {result['sources']}")

Question: What is the discount policy for furniture?
Answer: POLICY DOCUMENT: Discount Approval Policy Document ID: POL-SALES-004 Effective Date: January 1, 2024 Department: Sales Operations Purpose: This policy defines the approval thresholds for discounts applied to customer orders across all regions (West, East, South, Central). Policy Details: 1. Discounts under 10%: Can be approved directly by the sales representative handling the order, no further authorization needed. 2. Discounts between 10% and 20%: Require approval from the Regional Sales Manager before being applied to the order. 3. Discounts above 20%: Require written approval from the Director of Sales, and must be documented with a business justification. 4. Furniture category orders: Due to historically low profit margins in this category (particularly Tables and Bookcases), any discount above 5% requires Regional Sales Manager approval regardless of the order size. 5. All approved discounts must be logged in the order m

In [8]:
def extract_best_sentence(question, text, vectorizer):
    """
    Within a chunk, find the single sentence most relevant to the question
    using the same TF-IDF approach, but at sentence level.
    """
    # Split the chunk into sentences (simple split on periods)
    sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if s.strip()]
    
    if len(sentences) <= 1:
        return text  # too short to split further
    
    # Build a small TF-IDF just for these sentences
    sentence_vectorizer = TfidfVectorizer()
    sentence_matrix = sentence_vectorizer.fit_transform(sentences)
    question_vector = sentence_vectorizer.transform([question])
    
    similarities = cosine_similarity(question_vector, sentence_matrix)[0]
    best_idx = similarities.argmax()
    
    return sentences[best_idx] + "."

In [9]:
def ask_rag_extractive(question, vectorizer, tfidf_matrix, chunks, k=1):
    retrieved = retrieve_top_k(question, vectorizer, tfidf_matrix, chunks, k=k)
    
    if retrieved[0]['similarity_score'] < 0.05:
        return {
            "question": question,
            "answer": "I don't have enough information to answer that.",
            "sources": []
        }
    
    best = retrieved[0]
    best_sentence = extract_best_sentence(question, best['chunk']['text'], vectorizer)
    
    return {
        "question": question,
        "answer": best_sentence,
        "sources": [best['chunk']['source_document']],
        "confidence_score": best['similarity_score']
    }

In [10]:
result = ask_rag_extractive("What is the discount policy for furniture?", vectorizer, tfidf_matrix, chunks)

print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")
print(f"Source: {result['sources']}")
print(f"Confidence: {result['confidence_score']}")

Question: What is the discount policy for furniture?
Answer: All approved discounts must be logged in the order management system with the approver's name and the reason for the discount.
Source: ['policy_discount_approval.txt']
Confidence: 0.3314


In [11]:
def ask_rag_extractive(question, vectorizer, tfidf_matrix, chunks, k=1):
    """
    Returns the full most-relevant chunk as the answer.
    Simpler and more reliable than sentence-level extraction for short documents.
    """
    retrieved = retrieve_top_k(question, vectorizer, tfidf_matrix, chunks, k=k)
    
    if retrieved[0]['similarity_score'] < 0.05:
        return {
            "question": question,
            "answer": "I don't have enough information to answer that.",
            "sources": []
        }
    
    best = retrieved[0]
    return {
        "question": question,
        "answer": best['chunk']['text'],
        "sources": [best['chunk']['source_document']],
        "confidence_score": best['similarity_score']
    }

In [12]:
result = ask_rag_extractive("What is the discount policy for furniture?", vectorizer, tfidf_matrix, chunks)

print(f"Question: {result['question']}")
print(f"Answer: {result['answer']}")
print(f"Source: {result['sources']}")
print(f"Confidence: {result['confidence_score']}")

Question: What is the discount policy for furniture?
Answer: POLICY DOCUMENT: Discount Approval Policy Document ID: POL-SALES-004 Effective Date: January 1, 2024 Department: Sales Operations Purpose: This policy defines the approval thresholds for discounts applied to customer orders across all regions (West, East, South, Central). Policy Details: 1. Discounts under 10%: Can be approved directly by the sales representative handling the order, no further authorization needed. 2. Discounts between 10% and 20%: Require approval from the Regional Sales Manager before being applied to the order. 3. Discounts above 20%: Require written approval from the Director of Sales, and must be documented with a business justification. 4. Furniture category orders: Due to historically low profit margins in this category (particularly Tables and Bookcases), any discount above 5% requires Regional Sales Manager approval regardless of the order size. 5. All approved discounts must be logged in the order m

In [13]:
test_questions = [
    "What is the discount policy for furniture?",
    "How much was the total invoice amount?",
    "Which region performed best in October?",
    "What happens if a payment is late?",
    "How long does it take to process a new order?",
    "What payment methods are accepted?",
    "What discount was applied to the November invoice?",
    "Which sub-category had a negative profit margin?",
    "Who needs to approve a 15% discount?",
    "What is the capital of France?"  # intentionally unrelated question
]

evaluation_results = []

for q in test_questions:
    result = ask_rag_extractive(q, vectorizer, tfidf_matrix, chunks)
    evaluation_results.append({
        "question": q,
        "source": result['sources'][0] if result['sources'] else "NONE",
        "confidence": result.get('confidence_score', 0)
    })

import pandas as pd
eval_df = pd.DataFrame(evaluation_results)
eval_df

,question,source,confidence
0,What is the discount policy for furniture?,policy_discount_approval.txt,0.3314
1,How much was the total invoice amount?,invoice_nov2024.txt,0.2260
2,Which region performed best in October?,report_october2024.txt,0.2553
3,What happens if a payment is late?,faq_orders_payment.txt,0.3190
4,How long does it take to process a new order?,faq_orders_payment.txt,0.3469
5,What payment methods are accepted?,faq_orders_payment.txt,0.3207
6,What discount was applied to the November invo...,invoice_nov2024.txt,0.2813
7,Which sub-category had a negative profit margin?,report_october2024.txt,0.4280
8,Who needs to approve a 15% discount?,invoice_nov2024.txt,0.2218
9,What is the capital of France?,faq_orders_payment.txt,0.3339
